# Laboratorio de regresión logística

|                |   |
:----------------|---|
| **Nombre**     |   |
| **Fecha**      |   |
| **Expediente** |   | 

In machine learning, Support Vector Machines (SVM) are supervised learning models with associated learning algorithms that analyze data used for classification and regression analysis. It is mostly used in classification problems. In this algorithm, each data item is plotted as a point in p-dimensional space (where p is the number of features), with the value of each feature being the value of a particular coordinate. Then, classification is performed by finding the hyper-plane that best differentiates the two classes (or more if we have a multi class problem):

$$ f(x) = w^T \varphi(x) + b $$

where $\varphi: X \rightarrow F $ is a function that makes each input point $x$ correspond to a point in F, where F is a Hilbert space.

In addition to performing linear classification, SVMs can efficiently perform a non-linear classification, implicitly mapping their inputs into high-dimensional feature spaces (more specifically using the kernel trick, like the RBF funcion). 

[1]

OLS utilizes the squared residuals to fit the parameters. Large residuals caused by outliers may worsen the accuracy significantly.

Support Vectors use piecewise linear functions to counter this, in which a hyperparameter  $\epsilon$ called the margin lets errors that are less or equal to it be 0, and error larger than it be $e - \epsilon$. 

The problem to solve is:

\begin{split}
        \min_{w, b, \xi, \xi^*} \mathcal{P}_\epsilon(w, b, \xi) &= \frac{1}{2} w^T w + c \sum_{k=1}^{N} \xi_k \\
        \text{s.t. } & y_k [w^T \varphi(x_k) - b] \geq 1- \xi_k,\ \ k = 1, ..., N \\
        & \xi_k \geq 0,\ \ k = 1, ..., N
\end{split}


The most important question that arises when using a SVM is how to choose the correct hyperplane. Consider the following scenarios:

### Scenario 1

In this scenario there are three hyperplanes called A, B, and C. Now, the problem is to identify the hyperplane which best differentiates the stars and the circles.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_21-2.png" alt="what image shows"></center>

In this case, hyperplane B separates the stars and the circle betters, hence it is the correct hyperplane.


### Scenario 2

Now take another scenario where all three hyperplanes are segregating classes well. The question that arises is how to choose the best hyperplane in this situation.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/SVM_4-2.png" alt="what image shows"></center>

In such scenarios, we calculate the margin (which is the distance between nearest data point and the hyperplane). The hyperplane with the largest margin will be considered as the correct hyperplane to classify the dataset.

Here C has the largest margin. Hence, it is considered as the best hyperplane.


### Kernels
Knowing 
$$ w = \sum_{k=1}^{N} \alpha_k y_k \varphi(x_k) $$

And
$$ y_{pred} = w^T \varphi(x) + b $$

Then 
$$ y_{pred} = (\sum_{k=1}^{N} \alpha_k y_k \varphi(x_k))^T \varphi(x) + b $$

Where $\varphi$ is a function that makes each input in $x$ correspond to a point in $F$ (a Hilbert space). This can be seen as processing and transforming the input featuers to keep the model's convexity. [2]

This also allows us to transform the inputs into another space where they might be more easily classified.

<center><img src=https://miro.medium.com/max/838/1*gXvhD4IomaC9Jb37tzDUVg.png alt="what image shows"></center>

## ROC and AUC

A ROC (Receiver Operating Characteristic) is a graph that shows how the classification model performs at the classification thresholds. 

ROC curves typically feature true positive rate on the Y axis, and false positive rate on the X axis. This means that the top left corner of the plot is the “ideal” point - a false positive rate of zero, and a true positive rate of one. This is not very realistic, but it does mean that a larger area under the curve (AUC) is usually better. [3]

True Positive Rate is a synonym for Recall and defined as:
$$ TPR = \frac{TP}{TP + FN} $$

False Positive Rate is a synonym for Specificity and defined as:

$$ FPR = \frac{FP}{FP + TN} $$

ROC curves are typically used in binary classification to study the output of a classifier. In order to extend ROC curve and ROC area to multi-label classification, it is necessary to binarize the output. One ROC curve can be drawn per label, but one can also draw a ROC curve by considering each element of the label indicator matrix as a binary prediction (micro-averaging).

E.g. If you lower a classification threshold, more items would be classified as positive, increasing False Positives and True Positives.

AUC stands for Area under the ROC.

## Ejercicio 1

- Utiliza el dataset `Iris`, modela con SVC y haz Cross-Validation de diferentes kernels ('linear', 'poly', 'rbf', 'sigmoid').
- Modela con LogisticRegression.
- El método de Cross-Validation es K-Folds con $k=10$.
- Utiliza el AUC como métrico de Cross-Validation.
- Compara resultados.

### **Importaciones**

In [43]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, KFold


### **Cargar el dataset Iris**

In [44]:
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['Species'] = iris.target

df

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),Species
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


### **Separar variables 'X' y 'y'**

In [47]:
X = df[['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']]
y = df['Species']

### **Definir K-Fold (k = 10)**

In [48]:
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

### **Evaluar los diferentes kernels de SVC**

In [49]:
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
for k in kernels:
    model = SVC(kernel=k, probability=True)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=kfold,
        scoring='roc_auc_ovr'
    )

    print(f"Kernel: {k}")
    print("AUC promedio:", scores.mean())
    print("--------------------")


Kernel: linear
AUC promedio: 0.9981006647673315
--------------------
Kernel: poly
AUC promedio: 0.9956315289648623
--------------------
Kernel: rbf
AUC promedio: 0.996201329534663
--------------------
Kernel: sigmoid
AUC promedio: 0.7457193485526818
--------------------


### **Evaluar Logistic Regression**

In [50]:
log_model = LogisticRegression(max_iter=1000)

scores_log = cross_val_score(
    log_model,
    X,
    y,
    cv=kfold,
    scoring='roc_auc_ovr'
)

print("Logistic Regression AUC:", scores_log.mean())

Logistic Regression AUC: 0.9981006647673315


In [51]:
modelos = []
aucs = []

for k in kernels:
    model = SVC(kernel=k, probability=True)
    scores = cross_val_score(
        model,
        X,
        y,
        cv=kfold,
        scoring='roc_auc_ovr'
    )
    modelos.append(f'SVC {k}')
    aucs.append(scores.mean())

log_model = LogisticRegression(max_iter=1000)
scores_log = cross_val_score(
    log_model,
    X,
    y,
    cv=kfold,
    scoring='roc_auc_ovr'
)
modelos.append('Logistic Regression')
aucs.append(scores_log.mean())

resultados_df = pd.DataFrame({
    'Modelo': modelos,
    'AUC': aucs
})

resultados_df = resultados_df.sort_values(by='AUC', ascending=False)
resultados_df


,Modelo,AUC
0,SVC linear,0.998101
4,Logistic Regression,0.998101
2,SVC rbf,0.996201
1,SVC poly,0.995632
3,SVC sigmoid,0.728196


Al comparar los modelos se puede ver que SVC con kernel lineal y Logistic Regression obtuvieron el AUC promedio más alto, significa que son los que mejor logran distinguir entre las clases del dataset. Los kernels poly y rbf también tuvieron resultados bastante acertados, aunque ligeramente por debajo de los mejores modelos. En cambio, el kernel sigmoid presentó un resultado mucho menor, por lo que no es una opción tan adecuada para este problema. En general, los resultados muestran que SVC con kernel lineal y Logistic Regression son los modelos que mejor se ajustan a este dataset y ofrecen el mejor desempeño de clasificación según la métrica AUC.

## Ejercicio 2
- Repite el ejercicio 1 con el dataset `Default`. Utiliza `default` como target.

In [55]:
df_default = pd.read_csv('Datos/Default.csv')

# Separar variables X y y
X = df_default[['student', 'balance', 'income']].copy()
y = df_default['default'].copy()

# Convertir variables categóricas a numéricas
X['student'] = X['student'].map({'No': 0, 'Yes': 1})
y = y.map({'No': 0, 'Yes': 1})

# Definir K-Fold con k = 10
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# Kernels a evaluar
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

# Guardar resultados
modelos = []
aucs = []

# Evaluar SVC con diferentes kernels
for k in kernels:
    model = SVC(kernel=k)   # quitamos probability=True
    scores = cross_val_score(
        model,
        X,
        y,
        cv=kfold,
        scoring='roc_auc'
    )
    modelos.append(f'SVC {k}')
    aucs.append(scores.mean())
    print(f'Kernel: {k}')
    print('AUC promedio:', scores.mean())
    print('--------------------')

# Evaluar Logistic Regression
log_model = LogisticRegression(max_iter=1000)
scores_log = cross_val_score(
    log_model,
    X,
    y,
    cv=kfold,
    scoring='roc_auc'
)

modelos.append('Logistic Regression')
aucs.append(scores_log.mean())

print('Logistic Regression AUC:', scores_log.mean())

Kernel: linear
AUC promedio: 0.9290240177872358
--------------------
Kernel: poly
AUC promedio: 0.8941770528429643
--------------------
Kernel: rbf
AUC promedio: 0.6683559551213774
--------------------
Kernel: sigmoid
AUC promedio: 0.5388521546033146
--------------------
Logistic Regression AUC: 0.9498515498875946


### **Tabla comparativa**

In [60]:
resultados_df = pd.DataFrame({
    'Modelo': modelos,
    'AUC promedio': aucs
})

resultados_df = resultados_df.sort_values(by='AUC promedio', ascending=False)
resultados_df = resultados_df.reset_index(drop=True)

resultados_df

,Modelo,AUC promedio
0,Logistic Regression,0.949852
1,SVC linear,0.929024
2,SVC poly,0.894177
3,SVC rbf,0.668356
4,SVC sigmoid,0.538852


Al comparar los modelos utilizando la métrica AUC, se observa que Logistic Regression obtuvo el mejor desempeño, con un AUC promedio cercano a 0.95, lo que indica una muy buena capacidad para distinguir entre los clientes que caen en default y los que no.

El modelo SVC con kernel lineal también mostró un buen desempeño, con un AUC cercano a 0.93, seguido por el kernel poly, que obtuvo un resultado ligeramente menor.

Por otro lado, los kernels rbf y sigmoid presentaron un desempeño considerablemente menor, siendo sigmoid el modelo con el AUC más bajo.

En general, los resultados indican que Logistic Regression y SVC con kernel lineal son los modelos que mejor se ajustan a este dataset, ofreciendo el mejor desempeño de clasificación según la métrica AUC.

# Addendum

Métricos disponibles para clasificación:
- ‘accuracy’
- ‘balanced_accuracy’
- ‘top_k_accuracy’
- ‘average_precision’
- ‘neg_brier_score’
- ‘f1’
- ‘f1_micro’
- ‘f1_macro’
- ‘f1_weighted’
- ‘f1_samples’
- ‘neg_log_loss’
- ‘precision’ etc.
- ‘recall’ etc.
- ‘jaccard’ etc.
- ‘roc_auc’
- ‘roc_auc_ovr’
- ‘roc_auc_ovo’
- ‘roc_auc_ovr_weighted’
- ‘roc_auc_ovo_weighted’
- ‘d2_log_loss_score’

# References

[1] Shigeo Abe.Support Vector Machines for Pattern Classification,2Ed.Springer-Verlag London,2010. ISBN978-1-84996-097-7. URLhttps://www.springer.com/gp/book/9781849960977.

[2] Johan A K Suykens, Tony Van Gestel, Jos De Brabanter, BartDe Moor, and Joos Vandewalle.Least Squares Support VectorMachines. World Scientific,2002. ISBN9789812381514. URLhttps://www.worldscientific.com/worldscibooks/10.1142/5089.

[3] Bradley, A. P. (1997). The use of the area under the ROC curve in the evaluation of machine learning algorithms. Pattern recognition, 30(7), 1145-1159. URL https://www.researchgate.net/post/how_can_I_interpret_the_ROC_curve_result